# 시계열 전처리: RevIN + 패치 시퀀스 생성 (Colab / 메모리 절약 버전)

**사전 준비 (Drive 업로드)**
```
내 드라이브/grad_project/data/kospi_valid.parquet
```

**메모리 절약 핵심**: 종목별로 시퀀스를 생성하고 즉시 HDF5 파일에 append  
→ RAM에 전체 데이터를 올리지 않음 (종목당 ~7MB만 사용)

## 0-A. Colab 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q tqdm pyarrow h5py

from pathlib import Path

DRIVE_ROOT    = Path('/content/drive/MyDrive/grad_project')
DATA_DIR      = DRIVE_ROOT / 'data'
OUT_DIR       = DRIVE_ROOT / 'data' / 'sequences'
OUT_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = DATA_DIR / 'kospi_valid.parquet'
h5_path      = OUT_DIR / 'sequences.h5'

assert parquet_path.exists(), f'파일 없음: {parquet_path}'
print(f'✓ 입력: {parquet_path}  ({parquet_path.stat().st_size/1e6:.1f} MB)')
print(f'✓ 출력: {h5_path}')

## 0-B. 라이브러리 & 설정

In [ ]:
import gc
import warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

# ── 핵심 하이퍼파라미터 ──────────────────────────────
SEQ_LEN   = 60
PRED_LEN  = 5
PATCH_LEN = 5
STRIDE    = 1

TRAIN_END = '2024-06-30'
VAL_END   = '2024-12-31'

PRICE_COLS = ['Adj_Close', 'Open', 'High', 'Low', 'Volume']
TECH_COLS  = [
    'SMA_5', 'SMA_20', 'SMA_60',
    'EMA_12', 'EMA_26',
    'MACD', 'MACD_signal', 'MACD_hist',
    'RSI_14', 'BB_width', 'Volume_ratio',
    'Return_1d', 'Return_5d',
    'Volatility_20d', 'ATR_14',
]
FEATURE_COLS = PRICE_COLS + TECH_COLS
N_FEATURES   = len(FEATURE_COLS)
N_PATCHES    = SEQ_LEN // PATCH_LEN

print(f'피처: {N_FEATURES}개  |  입력: {SEQ_LEN}일({N_PATCHES}패치)  |  예측: {PRED_LEN}일')

## 1. 데이터 로드

In [ ]:
df = pd.read_parquet(parquet_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['종목코드', 'Date']).reset_index(drop=True)

# 결측 처리
df[FEATURE_COLS] = (
    df.groupby('종목코드')[FEATURE_COLS]
    .transform(lambda g: g.ffill().bfill().fillna(0))
)

print(f'전체: {len(df):,}행 | {df["종목코드"].nunique()}종목')
print(f'기간: {df.Date.min().date()} ~ {df.Date.max().date()}')
print(f'피처 결측: {df[FEATURE_COLS].isnull().sum().sum()}')

## 2. 핵심 함수 정의

In [ ]:
class RevIN:
    def __init__(self, eps=1e-8):
        self.eps = eps

    def normalize(self, x):
        self.mean = np.nanmean(x, axis=0, keepdims=True)
        self.std  = np.nanstd(x,  axis=0, keepdims=True) + self.eps
        return (x - self.mean) / self.std


def make_sequences_for_stock(stock_df, feature_cols, seq_len, pred_len, stride=1):
    """
    단일 종목 → 시퀀스 배열 반환 (메모리 절약: 종목당 ~7MB)
    """
    arr   = stock_df[feature_cols].values.astype(np.float32)
    dates = stock_df['Date'].values
    ci    = feature_cols.index('Adj_Close')
    total = len(arr)
    n_seq = max(0, (total - seq_len - pred_len) // stride + 1)

    if n_seq == 0:
        return None

    X_buf    = np.empty((n_seq, seq_len, len(feature_cols)), dtype=np.float32)
    mean_buf = np.empty((n_seq, len(feature_cols)), dtype=np.float32)
    std_buf  = np.empty((n_seq, len(feature_cols)), dtype=np.float32)
    y_ret    = np.empty(n_seq, dtype=np.float32)
    y_dir    = np.empty(n_seq, dtype=np.int8)
    date_buf = np.empty(n_seq, dtype=dates.dtype)

    for i, start in enumerate(range(0, n_seq * stride, stride)):
        end      = start + seq_len
        revin    = RevIN()
        x_norm   = revin.normalize(arr[start:end].copy())
        p_now    = arr[end - 1, ci]
        p_fut    = arr[end + pred_len - 1, ci]
        ret      = (p_fut - p_now) / (p_now + 1e-8)

        X_buf[i]    = x_norm
        mean_buf[i] = revin.mean[0]
        std_buf[i]  = revin.std[0]
        y_ret[i]    = ret
        y_dir[i]    = int(ret > 0)
        date_buf[i] = dates[end - 1]

    return X_buf, y_ret, y_dir, mean_buf, std_buf, date_buf


print('함수 정의 완료')

## 3. HDF5 초기화 (빈 파일 생성)

In [ ]:
# 기존 파일이 있으면 삭제 후 재생성
if h5_path.exists():
    h5_path.unlink()
    print('기존 sequences.h5 삭제')

SHP_X    = (N_PATCHES, PATCH_LEN, N_FEATURES)
SHP_FLAT = (SEQ_LEN, N_FEATURES)
CHUNK    = 512  # 한 번에 쓰는 단위

with h5py.File(h5_path, 'w') as f:
    for split in ['train', 'val', 'test']:
        g = f.create_group(split)
        g.create_dataset('X',      shape=(0,*SHP_X),    maxshape=(None,*SHP_X),    dtype='float32', chunks=(CHUNK,*SHP_X))
        g.create_dataset('X_flat', shape=(0,*SHP_FLAT), maxshape=(None,*SHP_FLAT), dtype='float32', chunks=(CHUNK,*SHP_FLAT))
        g.create_dataset('y_ret',  shape=(0,),          maxshape=(None,),          dtype='float32', chunks=(CHUNK,))
        g.create_dataset('y_dir',  shape=(0,),          maxshape=(None,),          dtype='int8',    chunks=(CHUNK,))
        g.create_dataset('means',  shape=(0,N_FEATURES),maxshape=(None,N_FEATURES),dtype='float32', chunks=(CHUNK,N_FEATURES))
        g.create_dataset('stds',   shape=(0,N_FEATURES),maxshape=(None,N_FEATURES),dtype='float32', chunks=(CHUNK,N_FEATURES))

print('HDF5 초기화 완료:', h5_path)

## 4. 전체 종목 처리 → 즉시 HDF5에 기록

In [ ]:
%%time

train_end_dt = pd.Timestamp(TRAIN_END)
val_end_dt   = pd.Timestamp(VAL_END)

codes      = df['종목코드'].unique()
fail_codes = []
split_counts = {'train': 0, 'val': 0, 'test': 0}

def h5_append(ds, new_data):
    """HDF5 dataset에 행 단위로 append"""
    n_old = ds.shape[0]
    n_new = new_data.shape[0]
    ds.resize(n_old + n_new, axis=0)
    ds[n_old:] = new_data


with h5py.File(h5_path, 'a') as f:
    for code in tqdm(codes, desc='종목 처리'):
        stock_df = df[df['종목코드'] == code].reset_index(drop=True)
        result   = make_sequences_for_stock(
            stock_df, FEATURE_COLS, SEQ_LEN, PRED_LEN, STRIDE
        )

        if result is None:
            fail_codes.append(code)
            continue

        X_flat, y_ret, y_dir, means, stds, seq_dates = result

        # 패치 변환 (메모리 절약: 인플레이스)
        X_patch = X_flat.reshape(len(X_flat), N_PATCHES, PATCH_LEN, N_FEATURES)

        seq_dates_pd = pd.to_datetime(seq_dates)
        masks = {
            'train': seq_dates_pd <= train_end_dt,
            'val':   (seq_dates_pd > train_end_dt) & (seq_dates_pd <= val_end_dt),
            'test':  seq_dates_pd > val_end_dt,
        }

        # 각 split에 즉시 기록
        for split, mask in masks.items():
            n = int(mask.sum())
            if n == 0:
                continue
            g = f[split]
            h5_append(g['X'],      X_patch[mask])
            h5_append(g['X_flat'], X_flat[mask])
            h5_append(g['y_ret'],  y_ret[mask])
            h5_append(g['y_dir'],  y_dir[mask])
            h5_append(g['means'],  means[mask])
            h5_append(g['stds'],   stds[mask])
            split_counts[split] += n

        # 명시적 메모리 해제
        del X_flat, X_patch, y_ret, y_dir, means, stds, seq_dates
        gc.collect()

print(f'\n처리 실패 종목: {len(fail_codes)}')
for split, cnt in split_counts.items():
    print(f'  {split:5s}: {cnt:>8,}개 시퀀스')

## 5. 검증

In [ ]:
with h5py.File(h5_path, 'r') as f:
    print(f'파일 크기: {h5_path.stat().st_size / 1e9:.2f} GB')
    print()
    for split in ['train', 'val', 'test']:
        g      = f[split]
        n      = g['X'].shape[0]
        x_shp  = g['X'].shape
        pos    = g['y_dir'][:].mean()
        nan_x  = np.isnan(g['X_flat'][:100]).sum()  # 앞 100개만 샘플 확인
        print(f'{split:5s} | {n:>8,}개 | X: {x_shp} | 상승: {pos:.3f} | NaN샘플: {nan_x}')

In [ ]:
# 타겟 분포 시각화
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
with h5py.File(h5_path, 'r') as f:
    for ax, split in zip(axes, ['train', 'val', 'test']):
        y_ret = f[split]['y_ret'][:]
        y_dir = f[split]['y_dir'][:]
        ax.hist(y_ret.clip(-0.15, 0.15), bins=100, color='steelblue', alpha=0.8)
        ax.axvline(0, color='red', lw=1.5, linestyle='--')
        ax.set_title(f'{split}  (N={len(y_ret):,}, 상승={y_dir.mean():.1%})')
        ax.set_xlabel(f'{PRED_LEN}일 수익률')
        ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
plt.suptitle('Split별 타겟 분포', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# feature 명세 저장
import pandas as pd
pd.DataFrame({
    'index':   range(N_FEATURES),
    'feature': FEATURE_COLS,
    'group':   ['price'] * len(PRICE_COLS) + ['technical'] * len(TECH_COLS),
}).to_csv(DATA_DIR / 'feature_meta.csv', index=False)

print('완료!')
print(f'  sequences.h5   → {h5_path}')
print(f'  feature_meta.csv → {DATA_DIR / "feature_meta.csv"}')
print()
print('▶ 다음 단계: FFT / DWT 주파수 도메인 특징 추출 (6월)')
print('  h5py로 X_flat을 배치 단위 로드 → FFT → 별도 h5에 저장')

## 6. (참고) PyTorch Dataset 예시

모델 학습 시 HDF5에서 배치 단위로 읽는 Dataset 클래스입니다.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class KospiH5Dataset(Dataset):
    """
    HDF5에서 배치 단위로 읽는 Dataset.
    파일을 항상 열어두지 않고 __getitem__ 시점에 열어 멀티프로세싱 안전.
    """
    def __init__(self, h5_path: str, split: str):
        self.h5_path = str(h5_path)
        self.split   = split
        with h5py.File(self.h5_path, 'r') as f:
            self.length = f[split]['X'].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.h5_path, 'r') as f:
            g     = f[self.split]
            X     = torch.tensor(g['X'][idx],     dtype=torch.float32)  # (n_patches, patch_len, n_features)
            y_ret = torch.tensor(g['y_ret'][idx], dtype=torch.float32)
            y_dir = torch.tensor(int(g['y_dir'][idx]), dtype=torch.long)
        return X, y_ret, y_dir


# 동작 확인
train_ds = KospiH5Dataset(h5_path, 'train')
loader   = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)

X_batch, y_ret_batch, y_dir_batch = next(iter(loader))
print(f'배치 X:     {X_batch.shape}')    # (64, 12, 5, 20)
print(f'배치 y_ret: {y_ret_batch.shape}') # (64,)
print(f'배치 y_dir: {y_dir_batch.shape}') # (64,)